# Phân tích kết quả huấn luyện Attention U-Net
Notebook này dùng để:
1. Vẽ đường cong Loss và Dice Score theo epoch
2. Xem ảnh so sánh kết quả từ evaluate.py
3. Kiểm tra model đã train

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
import sys

# Thêm thư mục gốc vào path để import được models và scripts
sys.path.append('..')

from models.attention_unet import AttentionUNet
from scripts.dataset import DentalDataset
from scripts.metrics import get_metrics

print('Import thành công!')

## 1. Kiểm tra model đã train

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Thiết bị: {device}')

model = AttentionUNet(n_classes=1).to(device)

model_path = '../models_saved/attention_unet_best.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f'Tải model thành công: {model_path}')
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Tổng tham số: {total_params:,}')
else:
    print(f'Chưa tìm thấy model tại: {model_path}')
    print('Hãy chạy train.py trước!')

## 2. Xem ảnh so sánh kết quả

In [ ]:
results_dir = '../results'
sample_files = sorted([f for f in os.listdir(results_dir) if f.startswith('sample_')])

print(f'Tìm thấy {len(sample_files)} ảnh so sánh')

# Hiển thị 5 ảnh đầu
fig, axes = plt.subplots(min(5, len(sample_files)), 1, figsize=(20, 5 * min(5, len(sample_files))))
if len(sample_files) == 1:
    axes = [axes]

for ax, fname in zip(axes, sample_files[:5]):
    img = mpimg.imread(os.path.join(results_dir, fname))
    ax.imshow(img)
    ax.set_title(fname, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Đánh giá thủ công trên vài ảnh test

In [ ]:
from torch.utils.data import DataLoader

test_ds = DentalDataset('../data/processed', split='test')
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

images, masks = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    preds = model(images)   # logit thô
    preds_prob = torch.sigmoid(preds)  # xác suất

# Vẽ 4 ảnh đầu
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    # Cột 1: ảnh gốc
    axes[i, 0].imshow(images[i, 0].cpu(), cmap='gray')
    axes[i, 0].set_title('Ảnh X-quang gốc')
    axes[i, 0].axis('off')
    
    # Cột 2: mask bác sĩ
    axes[i, 1].imshow(masks[i, 0].cpu(), cmap='gray')
    axes[i, 1].set_title('Mask bác sĩ (Ground truth)')
    axes[i, 1].axis('off')
    
    # Cột 3: mask AI
    pred_bin = (preds_prob[i, 0].cpu() > 0.5).float()
    axes[i, 2].imshow(pred_bin, cmap='gray')
    d, iou = get_metrics(preds[i:i+1], masks[i:i+1].to(device))
    axes[i, 2].set_title(f'Mask AI | Dice={d:.3f} IoU={iou:.3f}')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu: notebooks/visualization.png')

## 4. Đọc báo cáo metrics

In [ ]:
report_path = '../results/metrics_report.txt'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('Chưa có báo cáo. Hãy chạy evaluate.py trước!')